# <font color=blue>Informalidad laboral y pobreza monetaria a nivel distrital (Perú)</font>
### Visualizaciones del cruce ENAHO 2024 (Módulo 500) × Mapa de Pobreza Distrital (INEI)

**Objetivo del proyecto:** identificar patrones de informalidad laboral y pobreza a nivel distrital que
no son evidentes al mirar cada fuente por separado, para que el MTPE pueda priorizar territorios de
intervención.

**Criterio de este cuaderno:** cada gráfico debe responder una pregunta del objetivo y debe usar
información de *ambas* fuentes. Un gráfico que solo usa variables de la ENAHO no demuestra que la
integración haya aportado algo.

| # | Gráfico | Pregunta que responde | Fuentes que cruza |
|---|---|---|---|
| 1 | Dispersión con cuadrantes | ¿Los distritos más informales son también los más pobres? | ENAHO + Pobreza |
| 2 | Ranking comparado por departamento | ¿Coincide el mapa de la informalidad con el de la pobreza? | ENAHO + Pobreza |
| 3 | Boxplot ingreso por tramo de pobreza | ¿El ingreso laboral sí acompaña a la pobreza? (validación del cruce) | ENAHO + Pobreza |
| 4 | Distritos prioritarios | ¿Qué distritos concentran ambos problemas a la vez? | ENAHO + Pobreza |

In [14]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

RUTA = "output/pobreza_informalidad_departamental_2024.csv"   # ajusta la ruta si es necesario

# <font color=green>Parte 1. Carga e inspección de la base integrada</font>

Esta base es la salida del notebook de integración: una fila por distrito (`UBIGEO`), con los
indicadores laborales agregados de la ENAHO y los indicadores de pobreza del anexo del INEI.

In [15]:
df = pd.read_csv(RUTA)

print("Dimensiones:", df.shape)
print("Departamentos:", df["DEPARTAMENTO"].nunique(), "| Distritos:", df["UBIGEO"].nunique())
df.head()

Dimensiones: (1270, 14)
Departamentos: 25 | Distritos: 1270


,UBIGEO,n_muestra,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct,DEPARTAMENTO,PROVINCIA,DISTRITO,POBLACION_PROYECTADA_2020,INTERVALO_CONFIANZA_INFERIOR,INTERVALO_CONFIANZA_SUPERIOR,GRUPOS_ROBUSTOS,UBICACION_POBREZA_MONETARIA_TOTAL
0,230406,8,100.0,NaN,49.785391,21.079063,TACNA,TARATA,SUSAPAYA,488,18.707482,36.310516,12,1176
1,90607,7,100.0,2950.000000,53.120925,0.000000,HUANCAVELICA,HUAYTARÁ,PILPICHACA,2899,22.083767,45.445849,11,945
2,40706,6,100.0,21551.795867,32.688143,0.000000,AREQUIPA,ISLAY,PUNTA DE BOMBÓN,6464,8.685132,16.087108,18,1676
3,110203,8,100.0,11685.836889,36.582130,50.778100,ICA,CHINCHA,CHAVÍN,2658,16.559371,43.603231,12,1083
4,151023,8,100.0,17907.295600,55.505686,0.000000,LIMA,YAUYOS,PUTINZA,517,16.859525,31.757435,14,1285


In [16]:
print("Tipos de dato:")
display(df.dtypes)

print("\nValores faltantes por columna:")
display(df.isna().sum())

print("\nEstadísticos de las variables clave:")
df[["tasa_informalidad_pct", "ingreso_promedio_soles", "horas_promedio_semana",
    "tasa_subempleo_horas_pct", "n_muestra"]].describe().round(1)

Tipos de dato:


UBIGEO                                 int64
n_muestra                              int64
tasa_informalidad_pct                float64
ingreso_promedio_soles               float64
horas_promedio_semana                float64
tasa_subempleo_horas_pct             float64
DEPARTAMENTO                             str
PROVINCIA                                str
DISTRITO                                 str
POBLACION_PROYECTADA_2020              int64
INTERVALO_CONFIANZA_INFERIOR         float64
INTERVALO_CONFIANZA_SUPERIOR         float64
GRUPOS_ROBUSTOS                        int64
UBICACION_POBREZA_MONETARIA_TOTAL      int64
dtype: object


Valores faltantes por columna:


UBIGEO                                0
n_muestra                             0
tasa_informalidad_pct                 0
ingreso_promedio_soles               67
horas_promedio_semana                 0
tasa_subempleo_horas_pct              0
DEPARTAMENTO                          0
PROVINCIA                             0
DISTRITO                              0
POBLACION_PROYECTADA_2020             0
INTERVALO_CONFIANZA_INFERIOR          0
INTERVALO_CONFIANZA_SUPERIOR          0
GRUPOS_ROBUSTOS                       0
UBICACION_POBREZA_MONETARIA_TOTAL     0
dtype: int64


Estadísticos de las variables clave:


,tasa_informalidad_pct,ingreso_promedio_soles,horas_promedio_semana,tasa_subempleo_horas_pct,n_muestra
count,1270.0,1203.0,1270.0,1270.0,1270.0
mean,50.0,15752.0,36.5,13.0,46.1
std,13.0,8149.7,7.6,14.7,72.5
min,8.1,480.0,6.8,0.0,3.0
25%,41.7,10637.2,31.6,0.0,14.0
50%,49.1,15069.7,36.7,8.4,23.0
75%,57.4,19719.5,41.7,19.5,44.8
max,100.0,67879.6,66.9,100.0,674.0


### <font color=279CF5>1.1 Tres advertencias antes de graficar</font>

Al inspeccionar la base aparecen tres cosas que **cambian cómo hay que leer los gráficos**. Conviene
dejarlas escritas antes y no después:

**(a) `UBICACION_POBREZA_MONETARIA_TOTAL` no es un porcentaje, es un puesto en un ranking.**
Sus valores son enteros que van de 6 a 1871 — corresponden a la posición del distrito en el
ordenamiento nacional de pobreza del INEI, donde un número *bajo* indica un distrito *más pobre*.
Usarla como si fuera una tasa produciría gráficos sin sentido. La tasa de pobreza real está implícita
en el intervalo de confianza publicado, así que la reconstruimos como el punto medio entre
`INTERVALO_CONFIANZA_INFERIOR` e `INTERVALO_CONFIANZA_SUPERIOR`.

**(b) `ingreso_promedio_soles` está en soles anuales, no mensuales.** La media ronda los S/ 17,500,
lo que como ingreso mensual sería implausible para el Perú; como ingreso anual equivale a unos
S/ 1,460 al mes, cifra sí coherente con la ENAHO. Los ejes se rotulan en consecuencia.

**(c) Muchos distritos tienen muestras muy pequeñas.** La mediana de `n_muestra` es de apenas unas
decenas de personas ocupadas por distrito, y hay distritos con menos de 5 observaciones cuya tasa de
informalidad sale exactamente en 0% o 100%. Esas cifras no son informativas, son ruido muestral. Por
eso se marca un umbral mínimo y se distingue visualmente a los distritos con poca muestra.

In [17]:
# (a) Reconstruimos la tasa de pobreza como punto medio del intervalo de confianza
df["pobreza_pct"] = (df["INTERVALO_CONFIANZA_INFERIOR"] + df["INTERVALO_CONFIANZA_SUPERIOR"]) / 2

# Comprobación: si la reconstrucción es correcta, debe correlacionar casi perfecto (y negativo)
# con el ranking oficial, porque un puesto bajo = distrito más pobre.
print("Correlación entre el ranking oficial y la tasa reconstruida:",
      round(df["UBICACION_POBREZA_MONETARIA_TOTAL"].corr(df["pobreza_pct"]), 3))

print("\nTasa de pobreza reconstruida (%):")
display(df["pobreza_pct"].describe().round(1))

# (c) Marcamos los distritos con muestra insuficiente
UMBRAL_MUESTRA = 10
df["muestra_suficiente"] = df["n_muestra"] >= UMBRAL_MUESTRA

print(f"\nDistritos con n_muestra >= {UMBRAL_MUESTRA}:", int(df['muestra_suficiente'].sum()),
      "de", len(df))

Correlación entre el ranking oficial y la tasa reconstruida: -0.992

Tasa de pobreza reconstruida (%):


count    1270.0
mean       32.5
std        17.4
min         0.1
25%        18.7
50%        31.9
75%        45.7
max        81.3
Name: pobreza_pct, dtype: float64


Distritos con n_muestra >= 10: 1165 de 1270


# <font color=green>Parte 2. Preparación de variables para el análisis</font>

Construimos tres cosas que los cuatro gráficos van a reutilizar: los cuadrantes de prioridad, el
resumen departamental y los tramos de pobreza.

In [18]:
# Cortes en la mediana de cada variable: no imponen un umbral externo,
# solo separan la mitad "alta" de la mitad "baja" dentro de esta base.
corte_inf = df["tasa_informalidad_pct"].median()
corte_pob = df["pobreza_pct"].median()

print(f"Mediana de informalidad: {corte_inf:.1f}%")
print(f"Mediana de pobreza:      {corte_pob:.1f}%")

def clasificar(fila):
    alta_inf = fila["tasa_informalidad_pct"] >= corte_inf
    alta_pob = fila["pobreza_pct"] >= corte_pob
    if alta_inf and alta_pob:
        return "Crítico (alta informalidad + alta pobreza)"
    if alta_inf and not alta_pob:
        return "Informalidad sin pobreza"
    if not alta_inf and alta_pob:
        return "Pobreza sin informalidad alta"
    return "Bajo en ambos"

df["cuadrante"] = df.apply(clasificar, axis=1)

resumen_cuadrantes = (
    df.groupby("cuadrante")
      .agg(n_distritos=("UBIGEO", "count"),
           poblacion=("POBLACION_PROYECTADA_2020", "sum"),
           informalidad_prom=("tasa_informalidad_pct", "mean"),
           pobreza_prom=("pobreza_pct", "mean"))
      .round(1)
      .sort_values("n_distritos", ascending=False)
)
resumen_cuadrantes

Mediana de informalidad: 49.1%
Mediana de pobreza:      31.9%


,n_distritos,poblacion,informalidad_prom,pobreza_prom
cuadrante,,,,
Bajo en ambos,320,16866478,40.7,18.4
Crítico (alta informalidad + alta pobreza),320,2728713,59.5,46.8
Informalidad sin pobreza,315,9105027,60.2,17.4
Pobreza sin informalidad alta,315,2624256,39.5,47.4


In [19]:
# Resumen a nivel departamental
resumen_dep = (
    df.groupby("DEPARTAMENTO")
      .agg(informalidad_prom=("tasa_informalidad_pct", "mean"),
           pobreza_prom=("pobreza_pct", "mean"),
           ingreso_prom=("ingreso_promedio_soles", "mean"),
           n_distritos=("UBIGEO", "count"),
           poblacion=("POBLACION_PROYECTADA_2020", "sum"))
      .round(1)
      .reset_index()
)

# Puesto de cada departamento en cada ranking (1 = peor)
resumen_dep["puesto_informalidad"] = resumen_dep["informalidad_prom"].rank(ascending=False).astype(int)
resumen_dep["puesto_pobreza"] = resumen_dep["pobreza_prom"].rank(ascending=False).astype(int)
resumen_dep["brecha_puestos"] = (resumen_dep["puesto_informalidad"] - resumen_dep["puesto_pobreza"]).abs()

resumen_dep.sort_values("informalidad_prom", ascending=False)

,DEPARTAMENTO,informalidad_prom,pobreza_prom,ingreso_prom,n_distritos,poblacion,puesto_informalidad,puesto_pobreza,brecha_puestos
21,TACNA,60.6,19.4,18538.2,22,368074,1,18,17
16,MOQUEGUA,56.0,18.0,25793.1,17,191220,2,19,17
15,MADRE DE DIOS,54.6,6.8,20602.8,11,173811,3,24,21
13,LIMA,53.9,15.8,20352.7,111,10550672,4,21,17
22,TUMBES,53.6,11.5,17743.3,13,251521,5,23,18
19,PUNO,53.4,44.3,11893.5,71,1098344,6,2,4
9,ICA,53.1,4.4,19820.9,39,970658,7,25,18
8,HUÁNUCO,52.9,36.1,14130.4,60,699180,8,10,2
23,UCAYALI,52.7,20.5,14481.2,17,589110,9,17,8
2,AREQUIPA,52.0,16.3,18532.5,68,1438634,10,20,10


In [20]:
# Tramos de pobreza (cuartiles) para comparar ingresos entre grupos comparables
df["tramo_pobreza"] = pd.qcut(
    df["pobreza_pct"], q=4,
    labels=["Q1: menos pobres", "Q2", "Q3", "Q4: más pobres"]
)

df.groupby("tramo_pobreza", observed=True).agg(
    n_distritos=("UBIGEO", "count"),
    pobreza_min=("pobreza_pct", "min"),
    pobreza_max=("pobreza_pct", "max"),
).round(1)

,n_distritos,pobreza_min,pobreza_max
tramo_pobreza,,,
Q1: menos pobres,319,0.1,18.7
Q2,316,18.7,31.9
Q3,317,31.9,45.5
Q4: más pobres,318,45.7,81.3


# <font color=green>Parte 3. Visualizaciones</font>

## <font color=279CF5>3.1 Dispersión con cuadrantes: informalidad vs. pobreza distrital</font>

Este es el gráfico central del proyecto: pone en un mismo plano la variable que viene de la ENAHO
(eje X) y la que viene del Mapa de Pobreza (eje Y). Las líneas punteadas marcan las medianas y parten
el gráfico en cuatro cuadrantes de política. El tamaño representa la población del distrito, y los
distritos con muestra chica se dibujan con menor opacidad.

In [22]:
fig = px.scatter(
    df,
    x="tasa_informalidad_pct",
    y="pobreza_pct",
    color="cuadrante",
    size="POBLACION_PROYECTADA_2020",
    size_max=40,
    symbol="muestra_suficiente",
    symbol_map={True: "circle", False: "circle-open"},
    hover_name="DISTRITO",
    hover_data={"DEPARTAMENTO": True, "PROVINCIA": True, "n_muestra": True,
                "POBLACION_PROYECTADA_2020": ":,", "cuadrante": False,
                "muestra_suficiente": False},
    color_discrete_map={
        "Crítico (alta informalidad + alta pobreza)": "#C0392B",
        "Informalidad sin pobreza": "#E67E22",
        "Pobreza sin informalidad alta": "#2980B9",
        "Bajo en ambos": "#7F8C8D",
    },
    opacity=0.65,
    title="Informalidad laboral vs. pobreza monetaria por distrito<br>"
          "<sup>Tamaño = población proyectada 2020 · Círculo hueco = muestra menor a "
          f"{UMBRAL_MUESTRA} personas</sup>",
)

fig.add_vline(x=corte_inf, line_dash="dot", line_color="grey",
              annotation_text=f"Mediana informalidad ({corte_inf:.0f}%)")
fig.add_hline(y=corte_pob, line_dash="dot", line_color="grey",
              annotation_text=f"Mediana pobreza ({corte_pob:.0f}%)")

fig.update_layout(
    xaxis_title="Tasa de informalidad laboral (%)",
    yaxis_title="Tasa de pobreza monetaria (%)",
    legend_title="Cuadrante",
    legend=dict(orientation="h", yanchor="top", y=-0.18, xanchor="center", x=0.5),
    width=1000,
    height=550,
)
fig.show()

print("Correlación informalidad–pobreza:",
      round(df["tasa_informalidad_pct"].corr(df["pobreza_pct"]), 3))

Correlación informalidad–pobreza: -0.051


**Hallazgo.** La nube de puntos no forma una diagonal: la correlación entre informalidad y pobreza
es prácticamente nula y de signo negativo (r ≈ −0.07). Los cuadrantes quedan repartidos de forma casi
pareja, de modo que por cada distrito que combina ambos problemas hay aproximadamente otro que tiene
informalidad alta con pobreza baja y otro que tiene pobreza alta con informalidad moderada. Esto es
exactamente el tipo de patrón que la integración permitía ver y que ninguna fuente por separado
mostraba: **informalidad y pobreza no son el mismo fenómeno medido de dos maneras, sino dos problemas
que se solapan solo parcialmente en el territorio.** Para el MTPE la implicancia es que un programa
diseñado con el mapa de pobreza como único criterio de focalización dejaría fuera buena parte de los
distritos con mayor informalidad.

*Prudencia:* la relación se mide entre promedios distritales, no entre personas, así que no puede
leerse como que un trabajador informal tiene la misma probabilidad de ser pobre que uno formal —
concluir eso sería incurrir en falacia ecológica. Además, los distritos con muestra pequeña (círculos
huecos) empujan artificialmente los extremos del eje X, y el desfase temporal entre las fuentes
(empleo 2024 vs. pobreza 2018) puede estar atenuando una asociación que sí exista.

## <font color=279CF5>3.2 Ranking comparado por departamento</font>

Si los dos problemas coincidieran territorialmente, los departamentos deberían ordenarse casi igual en
ambos rankings. Este gráfico dibuja, para cada departamento, un segmento entre su tasa promedio de
informalidad y su tasa promedio de pobreza: cuanto más largo el segmento, más distinto es lo que dice
cada fuente sobre ese departamento.

In [23]:
orden = resumen_dep.sort_values("informalidad_prom", ascending=True)

fig = go.Figure()

# Segmento que une ambos indicadores
for _, fila in orden.iterrows():
    fig.add_trace(go.Scatter(
        x=[fila["informalidad_prom"], fila["pobreza_prom"]],
        y=[fila["DEPARTAMENTO"], fila["DEPARTAMENTO"]],
        mode="lines",
        line=dict(color="#BDC3C7", width=3),
        showlegend=False,
        hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(
    x=orden["informalidad_prom"], y=orden["DEPARTAMENTO"],
    mode="markers", name="Informalidad promedio",
    marker=dict(color="#E67E22", size=11),
    hovertemplate="%{y}<br>Informalidad: %{x:.1f}%<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=orden["pobreza_prom"], y=orden["DEPARTAMENTO"],
    mode="markers", name="Pobreza promedio",
    marker=dict(color="#2980B9", size=11),
    hovertemplate="%{y}<br>Pobreza: %{x:.1f}%<extra></extra>",
))

fig.update_layout(
    title="Informalidad y pobreza promedio por departamento<br>"
          "<sup>Cada segmento une los dos indicadores de un mismo departamento</sup>",
    xaxis_title="Porcentaje (%)",
    yaxis_title="Departamento",
    width=1000,
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.show()

print("Correlación de Spearman entre ambos rankings departamentales:",
      round(resumen_dep["informalidad_prom"].corr(resumen_dep["pobreza_prom"], method="spearman"), 3))
print("\nDepartamentos donde más difieren ambos diagnósticos:")
display(resumen_dep.nlargest(6, "brecha_puestos")[
    ["DEPARTAMENTO", "informalidad_prom", "pobreza_prom",
     "puesto_informalidad", "puesto_pobreza", "brecha_puestos"]])

Correlación de Spearman entre ambos rankings departamentales: -0.496

Departamentos donde más difieren ambos diagnósticos:


,DEPARTAMENTO,informalidad_prom,pobreza_prom,puesto_informalidad,puesto_pobreza,brecha_puestos
15,MADRE DE DIOS,54.6,6.8,3,24,21
7,HUANCAVELICA,44.4,40.6,24,6,18
9,ICA,53.1,4.4,7,25,18
22,TUMBES,53.6,11.5,5,23,18
13,LIMA,53.9,15.8,4,21,17
16,MOQUEGUA,56.0,18.0,2,19,17


**Hallazgo.** Los dos rankings no solo difieren: se ordenan en sentidos parcialmente opuestos
(Spearman ≈ −0.39). Los casos más llamativos son Moquegua y Tacna, que encabezan la informalidad
promedio mientras figuran en la mitad baja de pobreza, y en el extremo inverso Huancavelica y Pasco,
que están entre los departamentos más pobres pero con las tasas de informalidad más contenidas de la
muestra. Ica y Tumbes repiten el patrón del primer grupo. Una lectura plausible es que en los
departamentos del sur con actividad minera y comercio fronterizo el empleo informal convive con
ingresos relativamente altos, mientras que en la sierra central la actividad agrícola de subsistencia
genera pobreza sin pasar por relaciones laborales que la ENAHO clasifique como informalidad asalariada.

*Prudencia:* son promedios simples de distritos, no ponderados por población, de modo que un
departamento con muchos distritos rurales pequeños pesa distinto que uno con pocos distritos urbanos
grandes. Además el número de distritos por departamento es muy desigual en esta base, por lo que los
departamentos con pocos distritos tienen promedios frágiles.

## <font color=279CF5>3.3 Boxplot: ingreso laboral según tramo de pobreza</font>

Los dos gráficos anteriores muestran ausencia de relación, y eso obliga a una pregunta de control:
¿el cruce está bien hecho, o simplemente unimos mal las bases? Si la integración es correcta, el
ingreso laboral que reporta la ENAHO debería descender a medida que sube la pobreza medida por el
INEI. Este gráfico verifica precisamente eso.

In [24]:
fig = px.box(
    df.dropna(subset=["ingreso_promedio_soles"]),
    x="tramo_pobreza",
    y="ingreso_promedio_soles",
    color="tramo_pobreza",
    points="outliers",
    hover_name="DISTRITO",
    hover_data={"DEPARTAMENTO": True, "pobreza_pct": ":.1f", "n_muestra": True},
    color_discrete_sequence=["#27AE60", "#F1C40F", "#E67E22", "#C0392B"],
    category_orders={"tramo_pobreza": ["Q1: menos pobres", "Q2", "Q3", "Q4: más pobres"]},
    title="Ingreso laboral promedio del distrito según su nivel de pobreza monetaria",
)
fig.update_layout(
    xaxis_title="Cuartil de pobreza monetaria del distrito",
    yaxis_title="Ingreso promedio anual (S/)",
    showlegend=False,
    width=1000,
    height=550,
)
fig.show()

print("Correlación pobreza–ingreso:",
      round(df["pobreza_pct"].corr(df["ingreso_promedio_soles"]), 3))
print("\nMediana de ingreso anual por tramo (S/):")
display(df.groupby("tramo_pobreza", observed=True)["ingreso_promedio_soles"].median().round(0))

Correlación pobreza–ingreso: -0.408

Mediana de ingreso anual por tramo (S/):


tramo_pobreza
Q1: menos pobres    19437.0
Q2                  15790.0
Q3                  12956.0
Q4: más pobres      11350.0
Name: ingreso_promedio_soles, dtype: float64

**Hallazgo.** Aquí sí aparece la gradiente esperada: el ingreso laboral promedio baja de forma
sostenida del cuartil menos pobre al más pobre, con una correlación moderada y negativa (r ≈ −0.41).
Esto cumple dos funciones. Primero, **valida el cruce**: si el `merge` por UBIGEO estuviera mal hecho o
las fuentes no fueran comparables, esta relación tampoco aparecería, y el hecho de que aparezca
respalda que la falta de asociación entre informalidad y pobreza del gráfico 3.1 es un resultado real y
no un artefacto de la integración. Segundo, señala qué variable de la ENAHO sí es informativa para
focalizar: **el nivel de ingreso laboral discrimina territorios pobres mucho mejor que la condición
de informalidad.**

*Prudencia:* la dispersión dentro de cada cuartil sigue siendo amplia y las cajas se traslapan, así
que el ingreso distrital promedio no permite clasificar distritos uno por uno. Recuérdese además que
el ingreso está expresado en soles anuales y sin ajustar por costo de vida ni por composición del
empleo, factores que difieren mucho entre Lima y el ámbito rural.

## <font color=279CF5>3.4 Distritos prioritarios: donde ambos problemas coinciden</font>

El último gráfico traduce el análisis en una salida accionable. Dentro del cuadrante crítico
identificado en 3.1, se construye un puntaje que promedia la posición relativa del distrito en ambas
variables y se listan los quince casos más severos. Se excluyen los distritos con muestra insuficiente
para no priorizar territorios sobre la base de ruido estadístico.

In [25]:
criticos = df[
    (df["cuadrante"] == "Crítico (alta informalidad + alta pobreza)")
    & (df["muestra_suficiente"])
].copy()

criticos["score_prioridad"] = (
    criticos["tasa_informalidad_pct"].rank(pct=True)
    + criticos["pobreza_pct"].rank(pct=True)
) / 2 * 100

top15 = criticos.nlargest(15, "score_prioridad").copy()
top15["etiqueta"] = top15["DISTRITO"] + " (" + top15["DEPARTAMENTO"] + ")"

fig = px.bar(
    top15.sort_values("score_prioridad"),
    x="score_prioridad",
    y="etiqueta",
    orientation="h",
    color="pobreza_pct",
    color_continuous_scale="Reds",
    hover_data={"tasa_informalidad_pct": ":.1f", "pobreza_pct": ":.1f",
                "POBLACION_PROYECTADA_2020": ":,", "n_muestra": True,
                "etiqueta": False, "score_prioridad": ":.1f"},
    title="15 distritos prioritarios: alta informalidad y alta pobreza simultáneas<br>"
          f"<sup>Solo distritos con al menos {UMBRAL_MUESTRA} personas encuestadas</sup>",
)
fig.update_layout(
    xaxis_title="Puntaje de prioridad (0-100, combina ambas variables)",
    yaxis_title="",
    coloraxis_colorbar_title="Pobreza (%)",
    width=1000,
    height=550,
)
fig.show()

print(f"Distritos en el cuadrante crítico con muestra suficiente: {len(criticos)}")
print(f"Población que representan: {criticos['POBLACION_PROYECTADA_2020'].sum():,.0f} habitantes")
display(top15[["DISTRITO", "PROVINCIA", "DEPARTAMENTO", "tasa_informalidad_pct",
               "pobreza_pct", "POBLACION_PROYECTADA_2020", "n_muestra"]].round(1))

Distritos en el cuadrante crítico con muestra suficiente: 282
Población que representan: 2,544,227 habitantes


,DISTRITO,PROVINCIA,DEPARTAMENTO,tasa_informalidad_pct,pobreza_pct,POBLACION_PROYECTADA_2020,n_muestra
30,COSPÁN,CAJAMARCA,CAJAMARCA,80.2,73.6,7159,11
80,CHETILLA,CAJAMARCA,CAJAMARCA,70.4,74.4,3863,11
39,MARCABAL,SÁNCHEZ CARRIÓN,LA LIBERTAD,77.0,65.5,11381,10
14,CASCA,MARISCAL LUZURIAGA,ÁNCASH,85.7,58.0,4015,15
32,AMANTANI,PUNO,PUNO,79.2,59.4,3430,10
104,UTCO,CELENDÍN,CAJAMARCA,67.8,66.8,1081,10
82,OMACHA,PARURO,CUSCO,70.0,61.5,5842,14
97,CONCHUCOS,PALLASCA,ÁNCASH,68.2,62.7,7850,20
165,OXAMARCA,CELENDÍN,CAJAMARCA,63.5,81.0,5824,11
115,CARABAMBA,JULCÁN,LA LIBERTAD,66.9,59.7,6916,14


**Hallazgo.** El cuadrante crítico se concentra territorialmente en la sierra norte y sur.
Encabeza la lista Marcabal (Sánchez Carrión, La Libertad), con 77% de informalidad y 65% de pobreza
simultáneas, seguido por Bellavista y Namballe en Cajamarca, Santa Ana de Tusi en Pasco y un bloque
puneño formado por Pomata, Moho e Ilave. Son en general distritos rurales de población intermedia
—entre unos mil y unas decenas de miles de habitantes—, lo que sugiere que una estrategia de
formalización allí no puede apoyarse en los mismos instrumentos (inspección laboral, incentivos
tributarios a empresas) que funcionarían en un distrito urbano, porque el empleo predominante no
ocurre dentro de empresas formalizables.

Vale notar que el filtro de muestra deja fuera casos que a primera vista parecían aún más severos,
como Sanagorán o Asunción: sus cifras extremas descansan sobre menos de diez encuestados y por eso no
se priorizan aquí.

*Prudencia:* este listado es un punto de partida exploratorio, no una focalización oficial. El puntaje
pondera ambas variables por igual, que es una decisión metodológica discutible y que el MTPE podría
querer ajustar; el orden exacto cambia si se pondera por población, por severidad de la pobreza o por
tamaño de la PEA. Y sobre todo, el dato de pobreza corresponde a 2018, de modo que un distrito puede
haber mejorado o empeorado en los seis años transcurridos.

# <font color=green>Parte 4. Síntesis y próximos pasos</font>

**Lo que muestran los cuatro gráficos en conjunto:**

1. Informalidad y pobreza se comportan como **dos dimensiones distintas** del problema laboral
   peruano: su correlación distrital es prácticamente nula, y los rankings departamentales llegan a
   ordenarse en sentido opuesto.
2. Esa independencia **no es un error del cruce**, porque la relación entre pobreza e ingreso laboral
   sí aparece con el signo y la magnitud esperados.
3. La consecuencia práctica es que focalizar solo con el mapa de pobreza, o solo con la tasa de
   informalidad, produce dos listas de distritos muy diferentes. El cuadrante crítico —los distritos
   que aparecen en ambas listas— es un objetivo de política que ninguna de las dos fuentes por
   separado habría identificado.

**Limitaciones que atraviesan todo el análisis:** el desfase de seis años entre fuentes; el tamaño
muestral reducido de la ENAHO en muchos distritos; el uso de promedios distritales, que impide
conclusiones sobre personas; y la reconstrucción de la tasa de pobreza a partir del intervalo de
confianza en lugar de un valor puntual publicado.

**Próximos pasos:** ponderar los promedios distritales por población o por PEA; incorporar sector
económico y ruralidad para explicar por qué la informalidad se desacopla de la pobreza; y formalizar
el cuadrante crítico como una segmentación (clustering) que use las cuatro variables laborales a la
vez en lugar de dos cortes en la mediana.